# Day 27 — Merged Dashboard + Django REST Framework Fix

This notebook combines dashboard player search with the DRF `TemplateDoesNotExist: rest_framework/api.html` hotfix. It backs up settings/dashboard files, configures JSON responses, probes endpoints, and generates one documentation report.

In [1]:
from pathlib import Path
import os, sys, subprocess, shutil, datetime, re, json

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "django_api" / "manage.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find django_api/manage.py. Open this notebook from inside your IPL project.")
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
SETTINGS = DJANGO_ROOT / "config" / "settings.py"
if not SETTINGS.exists():
    raise FileNotFoundError(f"Django settings file not found: {SETTINGS}")
TEMPLATE = APP_DIR / "templates" / "ipl_api" / "dashboard_day25.html"
CSS = APP_DIR / "static" / "ipl_api" / "css" / "dashboard_day25.css"
JS = APP_DIR / "static" / "ipl_api" / "js" / "dashboard_day25.js"
URLS = APP_DIR / "urls.py"
DOCS = REPO_ROOT / "docs"
REPORTS = REPO_ROOT / "reports" / "day27"
DOCS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

for p in [TEMPLATE, CSS, JS, URLS]:
    print(("FOUND " if p.exists() else "MISSING "), p)
if not TEMPLATE.exists() or not JS.exists():
    raise FileNotFoundError("Day 25 dashboard template/JS not found. Run the Day 25 dashboard foundation first.")


FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\css\dashboard_day25.css
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\dashboard_day25.js
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py


## 1. Fix DRF template error
Back up settings, ensure `rest_framework` is in `INSTALLED_APPS`, and configure JSON-only rendering to avoid requiring the browsable API HTML template.

In [2]:
settings_before = SETTINGS.read_text(encoding="utf-8")
settings_source = settings_before
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
settings_backup = SETTINGS.with_name(f"settings.py.day27_merged_backup_{stamp}")
shutil.copy2(SETTINGS, settings_backup)
print("Settings backup:", settings_backup)

apps_match = re.search(r"(?ms)^INSTALLED_APPS\s*=\s*\[(.*?)^\]", settings_source)
if not apps_match:
    raise RuntimeError("Could not safely parse INSTALLED_APPS as a list; settings were not modified.")
apps_body = apps_match.group(1)
if not re.search(r"""[\'"]rest_framework[\'"]""", apps_body):
    settings_source = settings_source[:apps_match.start(1)] + apps_body + "\n    'rest_framework',\n" + settings_source[apps_match.end(1):]
    print("Added rest_framework to INSTALLED_APPS.")
else:
    print("rest_framework already in INSTALLED_APPS.")

renderer_block = """REST_FRAMEWORK = {
    "DEFAULT_RENDERER_CLASSES": [
        "rest_framework.renderers.JSONRenderer",
    ],
}"""
rf_match = re.search(r"(?ms)^REST_FRAMEWORK\s*=\s*\{.*?^\}", settings_source)
if rf_match:
    settings_source = settings_source[:rf_match.start()] + renderer_block + settings_source[rf_match.end():]
    print("Updated REST_FRAMEWORK renderer configuration.")
else:
    settings_source = settings_source.rstrip() + "\n\n" + renderer_block + "\n"
    print("Added JSON-only REST_FRAMEWORK renderer configuration.")

compile(settings_source, str(SETTINGS), "exec")
SETTINGS.write_text(settings_source, encoding="utf-8")
verified = SETTINGS.read_text(encoding="utf-8")
apps_verify = re.search(r"(?ms)^INSTALLED_APPS\s*=\s*\[(.*?)^\]", verified)
assert apps_verify and re.search(r"""[\'"]rest_framework[\'"]""", apps_verify.group(1))
assert "rest_framework.renderers.JSONRenderer" in verified
compile(verified, str(SETTINGS), "exec")
print("Verified: DRF app registered and JSONRenderer configured; settings syntax is valid.")


Settings backup: D:\Git\Personal\ipl-player-performance\django_api\config\settings.py.day27_merged_backup_20261023_214337
rest_framework already in INSTALLED_APPS.
Updated REST_FRAMEWORK renderer configuration.
Verified: DRF app registered and JSONRenderer configured; settings syntax is valid.


## 2. Inspect current URL configuration (read-only)

In [3]:
print("=== django_api/ipl_api/urls.py ===")
print(URLS.read_text(encoding="utf-8"))
PROJECT_URLS = DJANGO_ROOT / "config" / "urls.py"
print("\n=== django_api/config/urls.py ===")
print(PROJECT_URLS.read_text(encoding="utf-8") if PROJECT_URLS.exists() else "MISSING")


=== django_api/ipl_api/urls.py ===
from django.urls import path
from .eda_summary_day26 import eda_summary_day26
from .dashboard_views_day26 import dashboard_day25
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("eda/summary/", eda_summary_day26, name="eda-summary-day26"),
    path("dashboard/", dashboard_day25, name="ipl-dashboard"),
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]


=== django_api/config/urls.py ===
from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home-dashboard"),
    path(
        "",
        RedirectView.as_view(url="/api/dashboard/", permanent=False),
        name="home",
    ),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin

## 3. Update dashboard summary cards and player search

In [4]:
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
for path in [TEMPLATE, CSS, JS]:
    if path.exists():
        backup = path.with_name(path.name + f".day27_backup_{stamp}")
        shutil.copy2(path, backup)
        print("Backup:", backup)

template = r"""{% load static %}
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>IPL Player Analytics</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
  <link rel="stylesheet" href="{% static 'ipl_api/css/dashboard_day25.css' %}">
</head>
<body>
<nav class="navbar navbar-dark bg-dark px-3">
  <a class="navbar-brand" href="/">IPL Player Analytics</a>
  <span class="text-light small">Performance • EDA • Predictions</span>
</nav>
<main class="container-fluid py-4">
  <div class="d-flex flex-wrap justify-content-between align-items-center gap-2 mb-4">
    <div><h1 class="h3 mb-1">Player Performance Dashboard</h1><p class="text-secondary mb-0">Explore historical form and model predictions.</p></div>
    <button id="refreshButton" class="btn btn-primary">Refresh summary</button>
  </div>
  <div id="statusMessage" class="alert alert-secondary" role="status" aria-live="polite">Loading dashboard summary…</div>

  <div class="row g-3 mb-4">
    <div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Players</div><div class="metric-value" id="playersMetric">—</div></div></div></div>
    <div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Matches</div><div class="metric-value" id="matchesMetric">—</div></div></div></div>
    <div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Batting records</div><div class="metric-value" id="battingMetric">—</div></div></div></div>
    <div class="col-12 col-md-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Bowling records</div><div class="metric-value" id="bowlingMetric">—</div></div></div></div>
  </div>

  <section class="card mb-4">
    <div class="card-header fw-semibold">Player search</div>
    <div class="card-body">
      <form id="playerSearchForm" class="row g-2">
        <div class="col-12 col-md-8"><label class="visually-hidden" for="playerSearchInput">Player name</label><input id="playerSearchInput" class="form-control" type="search" placeholder="Enter player name (e.g. Virat)" minlength="1" maxlength="100"></div>
        <div class="col-12 col-md-4 d-grid"><button id="playerSearchButton" class="btn btn-outline-primary" type="submit">Search players</button></div>
      </form>
      <div id="playerSearchStatus" class="small text-secondary mt-3" role="status" aria-live="polite">Search uses the registered player API.</div>
      <div class="table-responsive mt-3">
        <table class="table table-striped table-hover align-middle mb-0">
          <thead><tr><th>Player</th><th>Details</th></tr></thead>
          <tbody id="playerResults"><tr><td colspan="2" class="text-secondary">Search for a player to see results.</td></tr></tbody>
        </table>
      </div>
    </div>
  </section>

  <div class="row g-3">
    <div class="col-12 col-lg-6"><div class="card h-100"><div class="card-header">API connection</div><div class="card-body"><p class="mb-1">Summary endpoint: <code>/api/eda/summary/</code></p><p class="small text-secondary mb-0">Summary is fetched automatically when the page loads and when Refresh summary is clicked.</p></div></div></div>
    <div class="col-12 col-lg-6"><div class="card h-100"><div class="card-header">Next integration steps</div><div class="card-body"><ul class="mb-0"><li>Verify registered batting/bowling EDA endpoints.</li><li>Connect prediction forms to validated model APIs.</li><li>Compare ML and DL metrics from saved evaluation reports.</li></ul></div></div></div>
  </div>
</main>
<script src="{% static 'ipl_api/js/dashboard_day25.js' %}"></script>
</body>
</html>
"""

css = r"""body { background: #f5f7fb; color: #172033; }
.metric-card { border: 0; box-shadow: 0 2px 12px rgba(25,35,60,.06); min-height: 118px; }
.metric-value { font-size: 2rem; font-weight: 700; margin-top: .25rem; }
.card { border-color: #e7eaf1; }
#playerResults td { overflow-wrap: anywhere; }
"""

js = r"""(() => {
  const $ = (id) => document.getElementById(id);
  const status = $("statusMessage");
  const summaryUrl = "/api/eda/summary/";
  const playerUrl = "/api/players/";

  async function getJson(url) {
    const response = await fetch(url, { headers: { "Accept": "application/json" } });
    const text = await response.text();
    let data;
    try { data = text ? JSON.parse(text) : {}; }
    catch (_) { throw new Error(`Expected JSON from ${url}; got a non-JSON response (HTTP ${response.status}).`); }
    if (!response.ok) {
      const detail = data.detail || data.error || data.message || `HTTP ${response.status}`;
      throw new Error(String(detail));
    }
    return data;
  }

  async function loadSummary() {
    status.className = "alert alert-info";
    status.textContent = "Loading summary…";
    try {
      const data = await getJson(summaryUrl);
      const aliases = {
        playersMetric: ["players", "player_count", "total_players"],
        matchesMetric: ["matches", "match_count", "total_matches"],
        battingMetric: ["batting_records", "batting_count"],
        bowlingMetric: ["bowling_records", "bowling_count"]
      };
      const missing = [];
      for (const [id, keys] of Object.entries(aliases)) {
        const key = keys.find(k => Object.prototype.hasOwnProperty.call(data, k));
        if (key) $(id).textContent = Number(data[key]).toLocaleString();
        else { $(id).textContent = "—"; missing.push(keys[0]); }
      }
      if (missing.length) {
        status.className = "alert alert-warning";
        status.textContent = `Summary API responded, but these fields were absent: ${missing.join(", ")}.`;
      } else {
        status.className = "alert alert-success";
        status.textContent = "Summary loaded successfully.";
      }
    } catch (err) {
      status.className = "alert alert-danger";
      status.textContent = `Could not load summary: ${err.message}`;
    }
  }

  function escapeHtml(value) {
    return String(value ?? "").replace(/[&<>"']/g, ch => ({
      "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"
    })[ch]);
  }

  function extractPlayers(data) {
    if (Array.isArray(data)) return data;
    if (data && Array.isArray(data.results)) return data.results;
    if (data && Array.isArray(data.players)) return data.players;
    if (data && Array.isArray(data.data)) return data.data;
    return null;
  }

  $("playerSearchForm").addEventListener("submit", async (event) => {
    event.preventDefault();
    const query = $("playerSearchInput").value.trim();
    const body = $("playerResults");
    const resultStatus = $("playerSearchStatus");
    if (!query) {
      resultStatus.textContent = "Enter a player name to search.";
      $("playerSearchInput").focus();
      return;
    }
    resultStatus.textContent = "Searching…";
    body.innerHTML = '<tr><td colspan="2">Loading…</td></tr>';
    $("playerSearchButton").disabled = true;
    try {
      const data = await getJson(`${playerUrl}?search=${encodeURIComponent(query)}&page=1&page_size=20`);
      const players = extractPlayers(data);
      if (players === null) {
        body.innerHTML = '<tr><td colspan="2">The API responded, but its JSON shape is not recognized. Inspect /api/players/ response.</td></tr>';
        resultStatus.textContent = "Search response format needs mapping.";
        return;
      }
      if (!players.length) {
        body.innerHTML = '<tr><td colspan="2">No matching players found.</td></tr>';
        resultStatus.textContent = "Search completed: no matches.";
        return;
      }
      body.innerHTML = players.map((p) => {
        const name = p.player_name ?? p.name ?? p.player ?? p.full_name ?? "Unnamed player";
        const details = Object.entries(p)
          .filter(([k, v]) => !["player_name", "name", "player", "full_name"].includes(k) && ["string", "number", "boolean"].includes(typeof v))
          .slice(0, 5)
          .map(([k, v]) => `${k}: ${v}`)
          .join(" • ");
        return `<tr><td>${escapeHtml(name)}</td><td>${escapeHtml(details || "Player record returned")}</td></tr>`;
      }).join("");
      resultStatus.textContent = `Found ${players.length} result(s).`;
    } catch (err) {
      body.innerHTML = '<tr><td colspan="2">Search failed. See the message above.</td></tr>';
      resultStatus.textContent = `Player search failed: ${err.message}. Confirm the endpoint supports the search parameter.`;
    } finally {
      $("playerSearchButton").disabled = false;
    }
  });

  $("refreshButton").addEventListener("click", loadSummary);
  // Automatically populate cards when the dashboard opens.
  loadSummary();
})(); 
"""

TEMPLATE.parent.mkdir(parents=True, exist_ok=True)
CSS.parent.mkdir(parents=True, exist_ok=True)
JS.parent.mkdir(parents=True, exist_ok=True)
TEMPLATE.write_text(template, encoding="utf-8")
CSS.write_text(css, encoding="utf-8")
JS.write_text(js, encoding="utf-8")
print("Updated dashboard template:", TEMPLATE)
print("Updated dashboard CSS:", CSS)
print("Updated dashboard JavaScript:", JS)


Backup: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html.day27_backup_20261023_214337
Backup: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\css\dashboard_day25.css.day27_backup_20261023_214337
Backup: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\dashboard_day25.js.day27_backup_20261023_214337
Updated dashboard template: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Updated dashboard CSS: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\css\dashboard_day25.css
Updated dashboard JavaScript: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\dashboard_day25.js


## 4. Run Django checks and probe endpoints as JSON

In [5]:
env = os.environ.copy()
check = subprocess.run([sys.executable, str(DJANGO_ROOT / "manage.py"), "check"],
                       cwd=str(DJANGO_ROOT), text=True, capture_output=True, env=env)
print("=== python manage.py check ===")
print("Return code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)

probe_script = r"""
from django.test import Client
c = Client()
for url in [
    "/api/eda/summary/",
    "/api/players/",
    "/api/players/?search=virat&page=1&page_size=20",
]:
    r = c.get(url, HTTP_ACCEPT="application/json")
    print("URL:", url)
    print("STATUS:", r.status_code)
    print("CONTENT_TYPE:", r.get("Content-Type"))
    print("BODY_PREVIEW:", getattr(r, "content", b"")[:1200].decode("utf-8", errors="replace"))
    print("---")
"""
probe = subprocess.run([sys.executable, str(DJANGO_ROOT / "manage.py"), "shell", "-c", probe_script],
                       cwd=str(DJANGO_ROOT), text=True, capture_output=True, env=env)
print("=== Endpoint probe ===")
print("Return code:", probe.returncode)
print(probe.stdout)
if probe.stderr:
    print(probe.stderr)


=== python manage.py check ===
Return code: 0
System check identified no issues (0 silenced).

=== Endpoint probe ===
Return code: 0
12 objects imported automatically (use -v 2 for details).

URL: /api/eda/summary/
STATUS: 400
CONTENT_TYPE: text/plain; charset=utf-8
BODY_PREVIEW: DisallowedHost at /api/eda/summary/
Invalid HTTP_HOST header: 'testserver'. You may need to add 'testserver' to ALLOWED_HOSTS.

Request Method: GET
Request URL: http://testserver/api/eda/summary/
Django Version: 6.0.9
Python Executable: c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe
Python Version: 3.12.14
Python Path: ['D:\\Git\\Personal\\ipl-player-performance\\django_api', 'c:\\Users\\Vinod\\anaconda3\\envs\\ipl-player-performance\\python312.zip', 'c:\\Users\\Vinod\\anaconda3\\envs\\ipl-player-performance\\DLLs', 'c:\\Users\\Vinod\\anaconda3\\envs\\ipl-player-performance\\Lib', 'c:\\Users\\Vinod\\anaconda3\\envs\\ipl-player-performance', 'c:\\Users\\Vinod\\anaconda3\\envs\\ipl-player-perfor

## 5. Generate merged documentation report

In [6]:
report = DOCS / "day27_merged_dashboard_api_report.md"
report_lines = [
    "# Day 27 — Merged Dashboard and DRF API Repair",
    "",
    f"- Generated: {datetime.datetime.now().isoformat(timespec='seconds')}",
    f"- Repository: `{REPO_ROOT}`",
    f"- Settings file: `{SETTINGS.relative_to(REPO_ROOT)}`",
    f"- Settings backup: `{settings_backup.relative_to(REPO_ROOT)}`",
    "",
    "## Dashboard files updated",
    f"- `{TEMPLATE.relative_to(REPO_ROOT)}`",
    f"- `{CSS.relative_to(REPO_ROOT)}`",
    f"- `{JS.relative_to(REPO_ROOT)}`",
    "",
    "## Checks",
    f"- Django system check return code: **{check.returncode}**",
    f"- Endpoint probe command return code: **{probe.returncode}**",
    "- DRF default renderer configured as JSONRenderer only to avoid the missing browsable API HTML template.",
    "",
    "## Endpoint probe output",
    "```text",
    (probe.stdout or probe.stderr or "(no output)").strip(),
    "```",
    "",
    "## Verification notes",
    "- A 200 JSON response confirms JSON API access, not necessarily search filtering.",
    "- Compare `/api/players/` and `?search=virat` results to verify search behavior.",
    "- Summary card counts are populated from the API response, not hardcoded.",
    "- Browser verification remains necessary at `http://127.0.0.1:8000/api/dashboard/`.",
]
report.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Documentation generated:", report)
print("Django check passed:", check.returncode == 0)
print("Probe command passed:", probe.returncode == 0)


Documentation generated: D:\Git\Personal\ipl-player-performance\docs\day27_merged_dashboard_api_report.md
Django check passed: True
Probe command passed: True


## 6. Browser verification checklist

1. Restart Django from `django_api`.
2. Open `http://127.0.0.1:8000/api/dashboard/` and confirm cards populate.
3. Open `http://127.0.0.1:8000/api/players/?search=virat&page=1&page_size=20` and confirm JSON is returned.
4. Compare filtered and unfiltered results; if search is ignored, update `PlayerListAPIView` based on the actual implementation and schema.